# 03 — Draft Model Prep
Préparation des features pour le modèle de recommandation de pick.
Ce notebook structure les données pour l'étape ML future.

**Objectif** : prédire le meilleur pick parmi ton pool étant donné :
- Le pick adverse mid
- La composition adverse (engage / assassin / poke / tank / wombo)
- Ton winrate récent sur chaque champion (rolling 20g)
- Les données de méta patch (tier, pick rate)

In [ ]:
import sys
sys.path.insert(0, '../..')

import pandas as pd
import numpy as np
from pipeline.load_db import get_games_as_df

df = get_games_as_df()
df = df.sort_values('game_date').reset_index(drop=True)
print(f'{len(df)} games loaded')
df[['champion_name', 'opponent_champion_name', 'win', 'patch']].head(10)

In [ ]:
# Feature 1: rolling winrate per champion (last 20 games)
MY_POOL = ['Orianna', 'Ahri', 'Galio', 'Mel', 'Anivia']

rolling_wr = {}
for champ in MY_POOL:
    champ_games = df[df['champion_name'] == champ].copy()
    champ_games['rolling_wr_20'] = champ_games['win'].rolling(20, min_periods=3).mean()
    if not champ_games.empty:
        latest = champ_games.iloc[-1]
        rolling_wr[champ] = {
            'games': len(champ_games),
            'rolling_wr_20': round(latest.get('rolling_wr_20', np.nan), 3),
            'overall_wr': round(champ_games['win'].mean(), 3),
        }

pd.DataFrame(rolling_wr).T

In [ ]:
# Feature 2: matchup winrate per (my_champ, opponent)
matchup_wr = df.groupby(['champion_name', 'opponent_champion_name']).agg(
    games=('win', 'count'),
    wr=('win', 'mean')
).reset_index()

# Only keep matchups with >=2 samples
matchup_wr = matchup_wr[matchup_wr['games'] >= 2]

# Pivot: rows = my_champ, cols = opponent
matchup_pivot = matchup_wr.pivot(index='champion_name', columns='opponent_champion_name', values='wr')

print(f'Matchup matrix shape: {matchup_pivot.shape}')
matchup_pivot.head()

In [ ]:
# Feature 3: scoring function (manual weights — to be replaced by ML)
# This replicates the Draft Helper logic from the Excel sheet, but in code.

COMP_PROFILES = {
    'full_engage':   {'Orianna': +3, 'Ahri':  0, 'Galio': +1, 'Mel': +1, 'Anivia':  0},
    'full_assassin': {'Orianna': -1, 'Ahri':  0, 'Galio': +3, 'Mel': +1, 'Anivia': -1},
    'full_poke':     {'Orianna':  0, 'Ahri': +2, 'Galio':  0, 'Mel': +2, 'Anivia': +1},
    'wombo_combo':   {'Orianna': +2, 'Ahri': +1, 'Galio':  0, 'Mel': +2, 'Anivia': +1},
    'tank_frontline':{'Orianna': +1, 'Ahri':  0, 'Galio': -1, 'Mel': +1, 'Anivia': +2},
    'full_ap':       {'Orianna':  0, 'Ahri':  0, 'Galio': +3, 'Mel': +1, 'Anivia':  0},
    'splitpush':     {'Orianna': +2, 'Ahri':  0, 'Galio': +1, 'Mel':  0, 'Anivia':  0},
}

WEIGHTS = {
    'matchup_wr': 0.40,
    'comp_score': 0.35,
    'rolling_wr': 0.25,
}

def recommend_pick(
    opponent_mid: str,
    comp_profile: str,  # key from COMP_PROFILES
    verbose: bool = True
) -> pd.DataFrame:
    scores = {}
    for champ in MY_POOL:
        # Matchup score (0 if unknown → assume 50%)
        try:
            mu_wr = matchup_wr[
                (matchup_wr['champion_name'] == champ) &
                (matchup_wr['opponent_champion_name'] == opponent_mid)
            ]['wr'].values[0]
        except IndexError:
            mu_wr = 0.5  # no data → neutral

        # Comp score (normalize to 0-1 range)
        raw_comp = COMP_PROFILES.get(comp_profile, {}).get(champ, 0)
        comp_score_norm = (raw_comp + 3) / 6  # range -3..+3 → 0..1

        # Rolling winrate
        rwr = rolling_wr.get(champ, {}).get('rolling_wr_20', 0.5) or 0.5

        total = (
            WEIGHTS['matchup_wr'] * mu_wr +
            WEIGHTS['comp_score'] * comp_score_norm +
            WEIGHTS['rolling_wr'] * rwr
        )

        scores[champ] = {
            'score': round(total, 3),
            'matchup_wr': round(mu_wr, 3),
            'comp_score_norm': round(comp_score_norm, 3),
            'rolling_wr': round(rwr, 3),
        }

    result = pd.DataFrame(scores).T.sort_values('score', ascending=False)
    if verbose:
        print(f'\nRecommendation vs {opponent_mid} | Compo: {comp_profile}')
        print(result)
        print(f'\n→ RECOMMENDED: {result.index[0]}')
    return result


# Example
recommend_pick('Zed', 'full_assassin')

In [ ]:
# TODO (phase ML) :
# 1. Collect 200+ games with opponent + team composition data
# 2. Encode team comp as bag-of-archetypes features
# 3. Train LogisticRegression / XGBoost with:
#    - Features: matchup_wr, comp_profile_vector, rolling_wr, patch_tier
#    - Target: win (binary)
# 4. Evaluate with rolling cross-validation (time-aware split)
# 5. Deploy as CLI tool or Discord bot command

print('Phase ML — à démarrer après 200+ games enregistrées.')
print(f'Données actuelles : {len(df)} games')
print(f'Pool games par champion :')
print(df[df['champion_name'].isin(MY_POOL)]['champion_name'].value_counts())